In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from utils import twh_to_ej_str, build_const_value_xml, build_const_techs_xml, write_text, twh_to_ej, xy, gw_to_twh
from pathlib import Path

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)
    * [Renewable energy capacity projection discussed at the Electricity Policy Council meeting](https://www.epj.co.kr/news/articleView.html?idxno=31848)
    * Ministry of Trade, Industry and Energy (MOTIE), 2024, Roadmap for Competitive Bidding in Offshore Wind Power (RCBOWP, `../resources/RCBOWP-20240320-MOTIE`)

* Implemented Input Files
    * `/input/policy/korea-2035/power/onwnid_const_value.xml`
    * `/input/policy/korea-2035/power/onwind_const_techs.xml`
    * `/input/policy/korea-2035/power/offwind_const_value_cp.xml`
    * `/input/policy/korea-2035/power/offwind_const_value_ep.xml`
    * `/input/policy/korea-2035/power/offwind_const_techs.xml`

# Wind

The 11th Basic Plan provides capacity targets for *total* wind power but does not separately report onshore and offshore targets. To disaggregate the total into onshore and offshore components, this study adopts offshore wind capacity milestones that were discussed at the Electricity Policy Council meeting in October 2022 during deliberations on the 10th Basic Plan: 0.1 GW (2023), 1.3 GW (2026), 14.3 GW (2030), 21.8 GW (2033), and 26.7 GW (2036). These offshore milestones are assumed to remain valid under the 11th Basic Plan.

Offshore capacity for intermediate model years is obtained by linear interpolation between the nearest milestone years (e.g., 2025 between 2023 and 2026; 2035 between 2033 and 2036). Onshore wind capacity is then calculated as the residual, defined as total wind capacity minus offshore wind capacity for each model period (2020, 2025, 2030, and 2035).

To translate capacity (GW) into electricity generation (TWh), the study derives an implicit capacity factor for each year from the Basic Plan’s total wind series, using

$ CF_y = \frac{G_y}{K_y \times 8.760} $,

where $ G_y $ denotes total wind generation (TWh) and $ K_y $ total wind capacity (GW). Offshore generation is computed as  

$ G^{off}_y = K^{off}_y \times 8.760 \times CF_y $,  

and onshore generation is defined as the residual  

$ G^{on}_y = G_y - G^{off}_y $,  

ensuring consistency with the total generation pathway.

In addition to the “current” offshore pathway, an “enhanced offshore” case is constructed by adding +20 GW by 2030 and +40 GW by 2035 relative to the interpolated 2025 offshore level, while retaining the same implicit capacity factors. These disaggregated onshore and offshore generation trajectories are then implemented in GCAM using technology-specific output constraints, expressed in energy units (EJ) for the model periods.


In [2]:
dictCapGWTot = {
    2020: 1.6, 2021: 1.7, 2022: 1.9, 2023: 2.2, 
    2024: 2.3, 2025: 3.0, 2026: 4.0, 2027: 6.0, 2028: 8.7, 2029: 14.7, 2030: 18.3,
    2031: 22.3, 2032: 24.9, 2033: 27.4, 2034: 30.0, 2035: 33.0, 2036: 35.5, 2037: 38.1, 2038: 40.7,
}
dictCapGWTot

{2020: 1.6,
 2021: 1.7,
 2022: 1.9,
 2023: 2.2,
 2024: 2.3,
 2025: 3.0,
 2026: 4.0,
 2027: 6.0,
 2028: 8.7,
 2029: 14.7,
 2030: 18.3,
 2031: 22.3,
 2032: 24.9,
 2033: 27.4,
 2034: 30.0,
 2035: 33.0,
 2036: 35.5,
 2037: 38.1,
 2038: 40.7}

In [3]:
dictCapGwOff = {2020: 0, 2023: 0.165, 2025: 2.27, 2030: 14.3, 2033: 21.8, 2036: 26.7}
# dictCapGwOff[2025] = dictCapGwOff[2023] + (dictCapGwOff[2026] - dictCapGwOff[2023]) * (2/3)
dictCapGwOff[2035] = dictCapGwOff[2033] + (dictCapGwOff[2036] - dictCapGwOff[2033]) * (2/3)
dictCapGwOff

{2020: 0,
 2023: 0.165,
 2025: 2.27,
 2030: 14.3,
 2033: 21.8,
 2036: 26.7,
 2035: 25.066666666666666}

In [4]:
(25.1 - 2.27) / 10

2.2830000000000004

In [5]:
2.3 * 0.46380 * 0.86,

(0.9173963999999998,)

In [6]:
dictGenGwOffEp = {
    2030: 14.3 * 0.45650 * 8.760,
    2035: 25.1 * 0.46030 * 8.760
}
dictGenGwOffEp

{2030: 57.184842, 2035: 101.2089228}

In [7]:
{y: twh_to_ej_str(dictGenGwOffEp[y]) for y in [2030, 2035]}

{2030: '0.206', 2035: '0.364'}

In [8]:
# target keys
years = list(range(2020, 2036, 5))  # [2020, 2025, 2030, 2035]

# compute onshore capacity
dictCapGwOn = {
    y: dictCapGWTot[y] - dictCapGwOff[y]
    for y in years
}

dictCapGwOn

{2020: 1.6, 2025: 0.73, 2030: 4.0, 2035: 7.933333333333334}

In [9]:
dictCapGenOnEp = {
    2030: 4.0 * 0.45018 * 8.760, 2035: 7.933333333333334 * 0.45293 * 8.760
}
{y: twh_to_ej_str(dictCapGenOnEp[y]) for y in [2030, 2035]}

{2030: '0.057', 2035: '0.113'}

In [10]:
dictGenTWhTot = {
    2020: 3.1, 2021: 3.2, 2022: 3.4, 2023: 3.4, 
    2024: 4.3, 2025: 5.3, 2026: 6.9, 2027: 10.4, 2028: 16.2, 2029: 26.9, 2030: 38.8,
    2031: 48.5, 2032: 56.9, 2033: 63.1, 2034: 69.4, 2035: 76.2, 2036: 83.2, 2037: 88.7, 2038: 94.4,
}

In [11]:
# Align years in case the dictionaries diverge later
years = sorted(set(dictCapGWTot) & set(dictGenTWhTot))
cap_gw = [dictCapGWTot[y] for y in years]
gen_twh = [dictGenTWhTot[y] for y in years]
cf = {y: dictGenTWhTot[y] / (dictCapGWTot[y] * 8.760) for y in years}
cap_factor = [cf[y] for y in years]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Capacity (GW)", "Generation (TWh)", "Implicit Capacity Factor")
)

fig.add_trace(go.Scatter(x=years, y=cap_gw, mode='lines+markers', name='Capacity'), row=1, col=1)
fig.add_trace(go.Scatter(x=years, y=gen_twh, mode='lines+markers', name='Generation'), row=1, col=2)
fig.add_trace(go.Scatter(x=years, y=cap_factor, mode='lines+markers', name='Capacity Factor'), row=1, col=3)

fig.update_yaxes(title_text='GW', row=1, col=1)
fig.update_yaxes(title_text='TWh', row=1, col=2)
fig.update_yaxes(title_text='Fraction', row=1, col=3)

fig.update_layout(
    template='plotly_white',
    width=1200, height=400,
    showlegend=False,
)

fig.show()


In [12]:
dictCapGwOffEp = {}
for year in [2020, 2023, 2025]:
    dictCapGwOffEp[year] = dictCapGwOff[year]
dictCapGwOffEp[2030] = 20 + dictCapGwOffEp[2025]
dictCapGwOffEp[2035] = 40 + dictCapGwOffEp[2025]

In [18]:
years = list(range(2020, 2036, 5))

dictGenTWhOff = {y: gw_to_twh(dictCapGwOff[y],   cf[y]) for y in years}
dictGenTWhOffEp = {y: gw_to_twh(dictCapGwOffEp[y], cf[y]) for y in years}

# Onshore generation is "residual" so totals remain consistent with your Totals series:
dictGenTWhOn = {y: dictGenTWhTot[y] - dictGenTWhOff[y] for y in years}
dictGenTWhOn[2025] = 3.49
dictGenTWhOnEp = {y: dictGenTWhTot[y] - dictGenTWhOffEp[y] for y in years}  # optional (onshore under enhanced)

COLOR_CURRENT = "royalblue"   # current + all onshore
COLOR_ENHANCED = "firebrick"  # enhanced offshore

# -----------------------------
# 5) Plot: 2x2 subplots (capacity + generation for on/off)
#     Offshore: show both current + enhanced
# -----------------------------
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Onshore Generation (TWh)",
        "Offshore Generation (TWh)",
    )
)


# Onshore generation (current)
fig.add_trace(go.Scatter(
    x=years, y=[dictGenTWhOn[y] for y in years],
    mode="lines+markers",
    name="Current",
    legendgroup="current",
    showlegend=False,         # already shown
    line=dict(color=COLOR_CURRENT),
), row=1, col=1)

# Offshore generation – current
fig.add_trace(go.Scatter(
    x=years, y=[dictGenTWhOff[y] for y in years],
    mode="lines+markers",
    name="Current",
    legendgroup="current",
    showlegend=False,
    line=dict(color=COLOR_CURRENT),
), row=1, col=2)

# Offshore generation – enhanced
fig.add_trace(go.Scatter(
    x=years, y=[dictGenTWhOffEp[y] for y in years],
    mode="lines+markers",
    name="Enhanced",
    legendgroup="enhanced",
    showlegend=False,         # already shown
    line=dict(color=COLOR_ENHANCED, dash="dash"),
), row=1, col=2)

fig.update_yaxes(title_text="TWh", row=1, col=1)
fig.update_yaxes(title_text="TWh", row=1, col=2)

fig.update_layout(
    template="plotly_white",
    width=1100, height=400,
    legend=dict(orientation="h", yanchor="bottom", xanchor="center", x=0.5),
)

import plotly.io as pio
pio.write_image(fig, "../figure/wind.jpg", width=1200, height=400, scale=3)
fig.show()

In [43]:
years_onshore = [2020, 2025, 2030, 2035]
years_offshore = [2025, 2030, 2035]
values_onshore = {y: twh_to_ej_str(dictGenTWhOn[y]) for y in years_onshore}
values_offshore_cp = {y: twh_to_ej_str(dictGenTWhOff[y]) for y in years_offshore}
values_offshore_ep = {y: twh_to_ej_str(dictGenTWhOffEp[y]) for y in years_offshore}
policy_name_onshore = "Onwind-Floor"
policy_name_offshore = "Offwind-Floor"
policy_type_onshore = "subsidy"
policy_type_offshore = 'subsidy'
subsector_name = 'wind'
tech_names_onshore = ['wind']
tech_names_offshore = ['wind_offshore']

In [44]:
xml_value_onshore = build_const_value_xml(
    values_by_year=values_onshore,
    policy_name=policy_name_onshore,
    policy_type=policy_type_onshore,
    # min_price=True,
    # min_price_values_by_year={2020: -10000, 2025: 0, 2030: 0, 2035: 0},
)

xml_techs_onshore = build_const_techs_xml(
    years=[2020, 2025, 2030, 2035],
    sector_name="electricity",
    subsector_name=subsector_name,
    policy_name=policy_name_onshore,
    tech_names=tech_names_onshore,
    policy_type=policy_type_onshore,
)

xml_value_offshore_cp = build_const_value_xml(
    values_by_year=values_offshore_cp,
    policy_name=policy_name_offshore,
    policy_type=policy_type_offshore,
)

xml_value_offshore_ep = build_const_value_xml(
    values_by_year=values_offshore_ep,
    policy_name=policy_name_offshore,
    policy_type=policy_type_offshore,
)

xml_techs_offshore = build_const_techs_xml(
    years=[2025, 2030, 2035],
    sector_name="electricity",
    subsector_name=subsector_name,
    policy_name=policy_name_offshore,
    tech_names=tech_names_offshore,
    policy_type=policy_type_offshore,
)

In [45]:
value_path_onshore = f"../../input/policy/korea-2035/power/onwind_const_value.xml"
techs_path_onshore = f"../../input/policy/korea-2035/power/onwind_const_techs.xml"

write_text(value_path_onshore, xml_value_onshore)
write_text(techs_path_onshore, xml_techs_onshore)

value_path_offshore_cp = f"../../input/policy/korea-2035/power/offwind_const_value_cp.xml"
value_path_offshore_ep = f"../../input/policy/korea-2035/power/offwind_const_value_ep.xml"
techs_path_offshore = f"../../input/policy/korea-2035/power/offwind_const_techs.xml"

write_text(value_path_offshore_cp, xml_value_offshore_cp)
write_text(value_path_offshore_ep, xml_value_offshore_ep)
write_text(techs_path_offshore, xml_techs_offshore)

print("Wrote:", Path(value_path_onshore).expanduser())
print("Wrote:", Path(techs_path_onshore).expanduser())
print("Wrote:", Path(value_path_offshore_cp).expanduser())
print("Wrote:", Path(value_path_offshore_ep).expanduser())
print("Wrote:", Path(techs_path_offshore).expanduser())

Wrote: ../../input/policy/korea-2035/power/onwind_const_value.xml
Wrote: ../../input/policy/korea-2035/power/onwind_const_techs.xml
Wrote: ../../input/policy/korea-2035/power/offwind_const_value_cp.xml
Wrote: ../../input/policy/korea-2035/power/offwind_const_value_ep.xml
Wrote: ../../input/policy/korea-2035/power/offwind_const_techs.xml
